# Inference walk-through

Quick interactive inference for the trained checkpoint, mirroring what `app.py` does under the hood.

Use this if you want to translate dozens of sentences from a notebook without spinning up Gradio.

In [ ]:
import torch
from tokenizers import Tokenizer

from config import get_best_weights_path, get_config
from decode import translate_text
from model import build_transformer

cfg = get_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tok_src = Tokenizer.from_file(cfg['tokenizer_file'].format(cfg['lang_src']))
tok_tgt = Tokenizer.from_file(cfg['tokenizer_file'].format(cfg['lang_tgt']))
model = build_transformer(
    tok_src.get_vocab_size(), tok_tgt.get_vocab_size(),
    cfg['max_seq_len'], cfg['max_seq_len'],
    d_model=cfg['d_model'], N=cfg['n_layers'], h=cfg['n_heads'],
    dropout=cfg['dropout'], d_ff=cfg['d_ff'],
).to(device)
state = torch.load(get_best_weights_path(cfg), map_location=device, weights_only=False)
model.load_state_dict(state['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (step={state.get('global_step')}, chrF++={state.get('best_chrf'):.2f})")

In [ ]:
EXAMPLES = [
    'Good morning, how are you today?',
    'The manufacture or processing of goods.',
    'Artificial intelligence is changing the world.',
    'She is reading a book in the library.',
    'India won the cricket match yesterday.',
    'Daniel Molkentin is a German software developer.',
]
for s in EXAMPLES:
    print(f'EN  ▶  {s}')
    print(f'HI  ▶  {translate_text(model, s, tok_src, tok_tgt, device, beam_size=4)}\n')

In [ ]:
# Greedy vs. beam side by side
S = 'Artificial intelligence is changing the world.'
print('greedy :', translate_text(model, S, tok_src, tok_tgt, device, beam_size=1))
print('beam 4 :', translate_text(model, S, tok_src, tok_tgt, device, beam_size=4))
print('beam 8 :', translate_text(model, S, tok_src, tok_tgt, device, beam_size=8))